# Phase 1 — Decoder-only Legal LM (v3 tokenizer)

Trains a Llama-style decoder (RMSNorm · RoPE · SwiGLU · QK-norm · tied embeddings) on the
`uint16` token files from **updated-tokenize-data**, using the tokenizer from **updated-bpe-tokenizer**,
then writes `prediction.json` + a submission zip.

**Inputs to attach on Kaggle:** both notebooks' outputs (auto-detected under `/kaggle/input`).
To resume in a new session, also attach this notebook's previous output — `last.pt` is picked up automatically.

## At a glance — Phase 1 · Architecture-1 — Llama-style decoder-only language model

| | |
|---|---|
| **Codabench** | BERTScore F1 **0.8238** (test continuations) |
| **Validation** | loss 1.0066 (perplexity 2.74) after 6,000 steps |
| **Size** | 109.5M parameters, 12 layers, width 768, context 2,048 tokens |

**In simple words:** the model reads legal text from left to right and learns one thing: **predict the next token**. After
≈ 393M tokens of training it continues a prompt with fluent legal text. It is the backbone most later phases start from
(Phases 2, 3, 5, 6 and 7).

**How it works:** decoder-only transformer with RMSNorm, rotary positions (RoPE), QK-norm, SwiGLU feed-forward layers and
tied input / output embeddings; AdamW with warm-up + cosine decay, float16 mixed precision, gradient accumulation; the four
corpus sources are mixed in fixed shares. Training can resume in a new Kaggle session from `last.pt`.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the libraries, fix the random seed and pick the GPU.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import sys
import json
import math
import time
import random
import zipfile

from pathlib import Path

import numpy as np

# less memory fragmentation on the GPU (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# Random seed
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device         :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU            :", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device         : cuda
GPU            : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting: model size, context 2,048, batch and accumulation (65,536 tokens per update), learning rate, 6,000 steps, and where to resume from.

**How it works:** 6,000 × 65,536 ≈ 393M training tokens.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Paths  (change these if your Kaggle input names are different)
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

TOKEN_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/"
    "legal_tokenized_100m"
)

OUTPUT_DIR = Path("/kaggle/working/legal_lm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# To continue training in a new Kaggle session, attach this notebook's
# previous output and put the path of its last.pt here (else keep None).
RESUME_FROM = None


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]

MIXING_RATIOS = {
    "acts": 0.15,
    "high_courts": 0.35,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}

BLOCK_SIZE = 2048     # context length
VAL_RATIO = 0.01      # 1% of blocks of every source are kept for validation


# ------------------------------------------------------------
# Model  (~110M parameters)
# ------------------------------------------------------------

MODEL_CONFIG = {
    "vocab_size": None,        # filled in from the tokenizer (32004)
    "block_size": BLOCK_SIZE,
    "d_model": 768,
    "n_layers": 12,
    "n_heads": 12,
    "d_ff": 2048,              # SwiGLU hidden size (≈ 8/3 × d_model)
    "rope_theta": 10000.0,
}


# ------------------------------------------------------------
# Training
#
# tokens per step = BATCH_SIZE × GRAD_ACCUM_STEPS × BLOCK_SIZE
#                 = 4 × 8 × 2048 = 65,536
# total tokens    = 9000 steps × 65,536 ≈ 590M
#
# If you get CUDA out-of-memory: BATCH_SIZE = 2, GRAD_ACCUM_STEPS = 16
# ------------------------------------------------------------

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
MAX_TRAIN_STEPS = 6000

LEARNING_RATE = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 300
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0

LOG_INTERVAL = 10
EVAL_INTERVAL = 250
MAX_VAL_BATCHES = 50

TIME_LIMIT_HOURS = 10.75    # stop + save before Kaggle's 12h limit
USE_COMPILE = False         # True = ~20-40% faster, but compiling takes a few minutes
NUM_WORKERS = 2


# ------------------------------------------------------------
# Debug mode: quick run to check the full pipeline
# ------------------------------------------------------------

DEBUG_MODE = False

if DEBUG_MODE:
    MAX_TRAIN_STEPS = 50
    EVAL_INTERVAL = 25
    MAX_VAL_BATCHES = 10
    DEBUG_TEST_EXAMPLES = 3


# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

GEN_TARGET_WORDS = 128
GEN_MAX_NEW_TOKENS = 768
GEN_TEMPERATURE = 0.8
GEN_TOP_K = 50
GEN_TOP_P = 0.95
GEN_REPETITION_PENALTY = 1.1


# ------------------------------------------------------------
# Mixed precision
#
# T4 / P100 do not support bfloat16 natively -> use float16 + GradScaler.
# A100 / H100 and newer -> bfloat16.
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


TOKENS_PER_STEP = BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE

print("Tokens per step  :", f"{TOKENS_PER_STEP:,}")
print("Total train tokens:", f"{TOKENS_PER_STEP * MAX_TRAIN_STEPS:,}")
print("AMP dtype        :", AMP_DTYPE if USE_AMP else "disabled")
print("Debug mode       :", DEBUG_MODE)

Tokens per step  : 65,536
Total train tokens: 393,216,000
AMP dtype        : torch.float16
Debug mode       : False


### CELL 3 · Load the tokenizer

**In simple words:** Load our BPE tokenizer and check an encode → decode round trip.

In [3]:
# ============================================================
# CELL 3 — LOAD TOKENIZER (same tokenizer.py as the tokenizer notebook)
# ============================================================

sys.path.insert(0, str(TOKENIZER_DIR))

from tokenizer import load_tokenizer

tokenizer, tokenizer_config = load_tokenizer(TOKENIZER_DIR)

VOCAB_SIZE = tokenizer.get_vocab_size()
MODEL_CONFIG["vocab_size"] = VOCAB_SIZE

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
BOS_ID = tokenizer.bos_id
EOS_ID = tokenizer.eos_id

print("Vocabulary size:", VOCAB_SIZE)
print("PAD / UNK / BOS / EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)


# ------------------------------------------------------------
# Sanity test
# ------------------------------------------------------------

test_text = "The Supreme Court held that the appeal was dismissed."

test_ids = tokenizer.encode(test_text)

print("\nText   :", test_text)
print("IDs    :", test_ids)
print("Tokens :", [tokenizer.id_to_token[i] for i in test_ids])
print("Decoded:", tokenizer.decode(test_ids))

assert tokenizer.decode(test_ids) == test_text

Vocabulary size: 32004
PAD / UNK / BOS / EOS: 0 1 2 3

IDs    : [1052, 12, 31296, 12, 1488, 12, 4473, 12, 9667, 12, 2207, 12, 20373, 12, 6430, 12, 9635, 16634, 36]
Tokens : ['The', ' ', 'Supreme', ' ', 'Court', ' ', 'held', ' ', 'that', ' ', 'the', ' ', 'appeal', ' ', 'was', ' ', 'dis', 'missed', '.']
Decoded: The Supreme Court held that the appeal was dismissed.


### CELL 4 · Load the tokenized data

**In simple words:** Open the four `uint16` token files *memory-mapped*: read from disk only when needed, so almost no RAM is used.

**How it works:** Output removed before publishing.

In [4]:
# ============================================================
# CELL 4 — LOAD TOKENIZED DATA (.bin files, uint16)
# ============================================================

with open(TOKEN_DIR / "tokenization_metadata.json", "r") as f:
    token_metadata = json.load(f)

assert token_metadata["tokenizer"]["vocab_size"] == VOCAB_SIZE, \
    "The .bin files were made with a different tokenizer!"


token_maps = {}

for source in SOURCES:

    bin_path = TOKEN_DIR / f"{source}_100m.bin"

    # memmap = read from disk only when needed (no big RAM use)
    tokens = np.memmap(bin_path, dtype=np.uint16, mode="r")

    token_maps[source] = tokens

    print(
        f"{source:15s} {len(tokens):>13,} tokens | "
        f"{len(tokens) // BLOCK_SIZE:>8,} blocks"
    )

print("\n✅ Tokenized data loaded.")

[output removed before publishing: this cell printed raw dataset / test text]


### CELL 5 · Train / validation split

**In simple words:** Cut every source into blocks of 2,048 tokens; about 1% of the blocks of every source are kept for validation.

**How it works:** A block needs 2,048 + 1 tokens: the input and the next-token targets.

In [5]:
# ============================================================
# CELL 5 — TRAIN / VALIDATION SPLIT (by blocks of BLOCK_SIZE)
# ============================================================

split_info = {}

for i, source in enumerate(SOURCES):

    # each block needs BLOCK_SIZE + 1 tokens (input + next-token target)
    num_blocks = (len(token_maps[source]) - 1) // BLOCK_SIZE

    indices = np.arange(num_blocks)

    rng = np.random.default_rng(SEED + i)
    rng.shuffle(indices)

    num_val = int(num_blocks * VAL_RATIO)

    split_info[source] = {
        "val": indices[:num_val],
        "train": indices[num_val:],
    }

    print(
        f"{source:15s} train={len(indices) - num_val:>8,} "
        f"val={num_val:>6,}"
    )

acts            train= 124,002 val= 1,252
high_courts     train= 117,154 val= 1,183
supreme_court   train= 116,741 val= 1,179
tribunals       train= 116,876 val= 1,180


### CELL 6 · Mixed training order

**In simple words:** Decide in advance which blocks every training step uses, mixing the four sources in fixed shares.

**How it works:** Resuming at step s simply skips the first s steps of this list — no data is seen twice.

In [6]:
# ============================================================
# CELL 6 — MIXED TRAINING ORDER + VALIDATION INDEX
#
# We build the full list of (source, block_id) that training will use,
# in order. One optimizer step uses BATCH_SIZE × GRAD_ACCUM_STEPS blocks.
# Resuming at step s simply skips the first s × BLOCKS_PER_STEP entries.
# ============================================================

BLOCKS_PER_STEP = BATCH_SIZE * GRAD_ACCUM_STEPS

total_blocks_needed = MAX_TRAIN_STEPS * BLOCKS_PER_STEP

rng = np.random.default_rng(SEED)

mixed_train_index = []

for source in SOURCES:

    available = split_info[source]["train"]

    if source == SOURCES[-1]:
        n = total_blocks_needed - len(mixed_train_index)   # exact total, no rounding gap
    else:
        n = int(total_blocks_needed * MIXING_RATIOS[source])

    # if a source is too small, some of its blocks are repeated
    repeat = n > len(available)

    chosen = rng.choice(available, size=n, replace=repeat)

    mixed_train_index += [(source, int(b)) for b in chosen]

    print(
        f"{source:15s} uses {n:>8,} blocks "
        f"({n / len(available):.0%} of its train blocks)"
    )

rng.shuffle(mixed_train_index)


val_index = [
    (source, int(b))
    for source in SOURCES
    for b in split_info[source]["val"]
]

np.random.default_rng(SEED + 999).shuffle(val_index)

print("\nTraining blocks  :", f"{len(mixed_train_index):,}")
print("Validation blocks:", f"{len(val_index):,}")

acts            uses   28,800 blocks (23% of its train blocks)
high_courts     uses   67,200 blocks (57% of its train blocks)
supreme_court   uses   48,000 blocks (41% of its train blocks)
tribunals       uses   48,000 blocks (41% of its train blocks)

Training blocks  : 192,000
Validation blocks: 4,794


### CELL 7 · Dataset + validation loader

**In simple words:** Every block becomes an (input, target) pair; the target is the input shifted by one token, so the model learns to predict the next token.

**How it works:** Output removed before publishing.

In [7]:
# ============================================================
# CELL 7 — DATASET + VALIDATION DATALOADER
# ============================================================

class LegalBlockDataset(Dataset):

    def __init__(self, index, token_maps, block_size):
        self.index = index
        self.token_maps = token_maps
        self.block_size = block_size

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):

        source, block_id = self.index[idx]

        start = block_id * self.block_size
        end = start + self.block_size + 1      # +1 for the target shift

        tokens = self.token_maps[source][start:end]
        tokens = torch.from_numpy(tokens.astype(np.int64))

        x = tokens[:-1]
        y = tokens[1:]

        return x, y


val_loader = DataLoader(
    LegalBlockDataset(val_index, token_maps, BLOCK_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

x, y = LegalBlockDataset(mixed_train_index, token_maps, BLOCK_SIZE)[0]

assert x.shape == (BLOCK_SIZE,) and y.shape == (BLOCK_SIZE,)
assert torch.equal(x[1:], y[:-1])
assert int(x.max()) < VOCAB_SIZE

print("x shape:", tuple(x.shape), "| y shape:", tuple(y.shape))
print("\nSample text from a training block:\n")
print(tokenizer.decode(x[:80].tolist()))

[output removed before publishing: this cell printed raw dataset / test text]


### CELL 8 · RMSNorm + RoPE

**In simple words:** Two building blocks: RMSNorm (a simpler LayerNorm) and RoPE, which tells attention *where* every token is by rotating its query and key vectors.

In [8]:
# ============================================================
# CELL 8 — RMSNORM + ROTARY POSITION EMBEDDING (RoPE)
# ============================================================

class RMSNorm(nn.Module):
    """LayerNorm without mean-centering and bias (used in Llama)."""

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms * self.weight).to(x.dtype)


def build_rope_cache(block_size, head_dim, theta):
    """Pre-compute cos / sin for every position: shape (block_size, head_dim / 2)."""

    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(block_size).float()

    angles = torch.outer(positions, inv_freq)

    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    """Rotate query / key vectors. x: (B, n_heads, T, head_dim)"""

    x1, x2 = x.float().chunk(2, dim=-1)

    rotated = torch.cat(
        [x1 * cos - x2 * sin,
         x1 * sin + x2 * cos],
        dim=-1,
    )

    return rotated.to(x.dtype)

### CELL 9 · Causal self-attention

**In simple words:** Attention: every token looks at the earlier tokens (never the later ones) and collects what is relevant.

**How it works:** 12 heads; RoPE; QK-norm keeps the attention scores stable in float16; PyTorch's fused attention kernel; a KV cache for fast generation.

In [9]:
# ============================================================
# CELL 9 — CAUSAL SELF-ATTENTION (with RoPE, QK-norm, KV cache)
# ============================================================

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]

        self.n_heads = config["n_heads"]
        self.head_dim = d_model // self.n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # normalizing q and k keeps attention stable in float16
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin, kv_cache=None):

        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        # (B, T, C) -> (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(self.q_norm(q), cos, sin)
        k = apply_rope(self.k_norm(k), cos, sin)

        # ----------------------------------------------------
        # KV cache (only used during generation):
        # keep keys / values of previous tokens so each new
        # token does not recompute the whole sequence
        # ----------------------------------------------------

        if kv_cache is not None:

            if kv_cache["k"] is not None:
                k = torch.cat([kv_cache["k"], k], dim=2)
                v = torch.cat([kv_cache["v"], v], dim=2)

            kv_cache["k"] = k
            kv_cache["v"] = v

        # T > 1  -> normal causal attention
        # T == 1 -> one new token attends to all cached tokens
        y = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(y)

### CELL 10 · SwiGLU feed-forward + transformer block

**In simple words:** The feed-forward layer and the block: attention, then feed-forward, each with a norm before it and a residual connection around it.

In [10]:
# ============================================================
# CELL 10 — SWIGLU FEED-FORWARD + TRANSFORMER BLOCK
# ============================================================

class SwiGLU(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.norm1 = RMSNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)

        self.norm2 = RMSNorm(config["d_model"])
        self.mlp = SwiGLU(config)

    def forward(self, x, cos, sin, kv_cache=None):

        # Pre-norm residual blocks
        x = x + self.attn(self.norm1(x), cos, sin, kv_cache)
        x = x + self.mlp(self.norm2(x))

        return x

### CELL 11 · Decoder-only language model

**In simple words:** The whole model: token embedding → 12 blocks → final norm → output layer that scores every possible next token.

**How it works:** The input embedding and the output layer share their weights (tying); careful initialization.

In [11]:
# ============================================================
# CELL 11 — DECODER-ONLY LANGUAGE MODEL
# ============================================================

class DecoderOnlyLM(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])

        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config["n_layers"])]
        )

        self.norm_f = RMSNorm(config["d_model"])

        self.lm_head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)

        # Weight tying: input embedding and output layer share weights
        self.lm_head.weight = self.token_embedding.weight

        # RoPE tables (not trainable, not saved in checkpoints)
        head_dim = config["d_model"] // config["n_heads"]
        cos, sin = build_rope_cache(config["block_size"], head_dim, config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # ----------------------------------------------------
        # Weight initialization
        # ----------------------------------------------------

        self.apply(self._init_weights)

        # smaller init for layers that write into the residual stream
        residual_std = 0.02 / math.sqrt(2 * config["n_layers"])

        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=residual_std)
            nn.init.normal_(block.mlp.down.weight, mean=0.0, std=residual_std)

    def _init_weights(self, module):

        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None, kv_caches=None, start_pos=0):

        B, T = input_ids.shape

        assert start_pos + T <= self.config["block_size"], "sequence longer than block_size"

        # RoPE values for positions start_pos ... start_pos + T - 1
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.token_embedding(input_ids)

        for i, block in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches is not None else None
            x = block(x, cos, sin, cache)

        x = self.norm_f(x)

        # ----------------------------------------------------
        # Training: return the loss
        # ----------------------------------------------------

        if targets is not None:

            logits = self.lm_head(x)

            loss = F.cross_entropy(
                logits.float().view(-1, logits.size(-1)),
                targets.view(-1),
            )

            return loss

        # ----------------------------------------------------
        # Generation: return logits of the last position only
        # ----------------------------------------------------

        return self.lm_head(x[:, -1, :])

### CELL 12 · Create + verify the model

**In simple words:** Build the model (109.5M parameters) and check one forward pass.

In [12]:
# ============================================================
# CELL 12 — CREATE + VERIFY MODEL
# ============================================================

# ------------------------------------------------------------
# Free GPU memory from a previous run of these cells
# (old model, optimizer, gradients, last error traceback)
# ------------------------------------------------------------

for name in ["model", "train_model", "optimizer", "scaler", "loss", "xb", "yb",
             "test_x", "test_y", "test_loss", "checkpoint", "train_iterator"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"GPU memory in use before creating model: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


model = DecoderOnlyLM(MODEL_CONFIG).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())

print("Model config:")
for key, value in MODEL_CONFIG.items():
    print(f"  {key:12s}: {value}")

print(f"\nTotal parameters: {total_params:,}")

assert 50_000_000 <= total_params <= 300_000_000, "Phase 1 requires 50M-300M parameters"


# ------------------------------------------------------------
# Short forward-pass test
# ------------------------------------------------------------

test_x = x[:128].unsqueeze(0).to(DEVICE)
test_y = y[:128].unsqueeze(0).to(DEVICE)

with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    test_loss = model(test_x, test_y)

# a random model should give loss ≈ ln(vocab_size) ≈ 10.4
print(f"Initial loss: {test_loss.item():.3f}  (expected ≈ {math.log(VOCAB_SIZE):.3f})")

GPU memory in use before creating model: 0.00 GB
Model config:
  vocab_size  : 32004
  block_size  : 2048
  d_model     : 768
  n_layers    : 12
  n_heads     : 12
  d_ff        : 2048
  rope_theta  : 10000.0

Total parameters: 109,534,464
Initial loss: 10.204  (expected ≈ 10.374)


### CELL 13 · Optimizer, schedule, grad scaler

**In simple words:** AdamW (no weight decay on norm weights); the learning rate warms up, then decays (cosine); the GradScaler keeps float16 gradients from becoming 0.

In [13]:
# ============================================================
# CELL 13 — OPTIMIZER + LR SCHEDULE + GRAD SCALER
# ============================================================

# no weight decay for norm weights (1-D parameters)
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=BETAS,
    eps=1e-8,
    fused=(DEVICE.type == "cuda"),   # faster AdamW on GPU
)

# GradScaler prevents float16 gradients from underflowing to zero
scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)


def get_lr(step):
    """Linear warmup, then cosine decay down to MIN_LR."""

    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / WARMUP_STEPS

    progress = (step - WARMUP_STEPS) / max(1, MAX_TRAIN_STEPS - WARMUP_STEPS)
    progress = min(1.0, progress)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

    return MIN_LR + (LEARNING_RATE - MIN_LR) * cosine


print("LR at step 0     :", f"{get_lr(0):.2e}")
print("LR after warmup  :", f"{get_lr(WARMUP_STEPS):.2e}")
print("LR at last step  :", f"{get_lr(MAX_TRAIN_STEPS - 1):.2e}")

LR at step 0     : 2.00e-06
LR after warmup  : 6.00e-04
LR at last step  : 6.00e-05


### CELL 14 · Validation + checkpoints

**In simple words:** Functions to measure the validation loss and save checkpoints (`best_model.pt`, `last.pt`).

In [14]:
# ============================================================
# CELL 14 — VALIDATION + CHECKPOINT FUNCTIONS
# ============================================================

BEST_CHECKPOINT = OUTPUT_DIR / "best_model.pt"
LAST_CHECKPOINT = OUTPUT_DIR / "last.pt"


@torch.no_grad()
def evaluate_model(model, loader, max_batches):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch_idx, (xb, yb) in enumerate(loader):

        if batch_idx >= max_batches:
            break

        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = model(xb, yb)

        total_loss += loss.item()
        total_batches += 1

    model.train()

    return total_loss / max(1, total_batches)


def save_checkpoint(path, step, best_val_loss):

    checkpoint = {
        "step": step,
        "best_val_loss": best_val_loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": MODEL_CONFIG,
    }

    torch.save(checkpoint, path)

    print(f"Checkpoint saved: {path}")

### CELL 15 · Resume

**In simple words:** If `last.pt` of an earlier session is attached, continue from it; otherwise start from scratch.

In [15]:
# ============================================================
# CELL 15 — RESUME FROM CHECKPOINT (if available)
# ============================================================

global_step = 0
best_val_loss = float("inf")


if LAST_CHECKPOINT.exists():
    checkpoint_path = LAST_CHECKPOINT            # same session
elif RESUME_FROM is not None:
    checkpoint_path = Path(RESUME_FROM)          # previous session
else:
    checkpoint_path = None


if checkpoint_path is not None:

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

    assert checkpoint["model_config"] == MODEL_CONFIG, \
        "Checkpoint was trained with a different MODEL_CONFIG"

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    global_step = checkpoint["step"]
    best_val_loss = checkpoint["best_val_loss"]

    del checkpoint

    print("✅ Resumed from:", checkpoint_path)
    print("Step          :", global_step)
    print("Best val loss :", best_val_loss)

else:

    print("Starting training from scratch.")

Starting training from scratch.


### CELL 16 · Training loop

**In simple words:** The training: 6,000 updates of next-token prediction, validation at regular intervals, the best model is saved.

**How it works:** Gradient accumulation, float16 autocast, gradient clipping, optional `torch.compile`.

In [16]:
# ============================================================
# CELL 16 — TRAINING LOOP
# ============================================================

# ------------------------------------------------------------
# Train loader starts where the previous run stopped
# ------------------------------------------------------------

start_block = global_step * BLOCKS_PER_STEP

train_loader = DataLoader(
    LegalBlockDataset(mixed_train_index[start_block:], token_maps, BLOCK_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,                 # already shuffled in CELL 6
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)

train_iterator = iter(train_loader)


# ------------------------------------------------------------
# Optional torch.compile (faster training on GPU)
# ------------------------------------------------------------

train_model = torch.compile(model) if USE_COMPILE else model


print("=" * 80)
print(f"TRAINING: step {global_step} -> {MAX_TRAIN_STEPS}")
print("=" * 80)

model.train()

session_start = time.time()
log_start = time.time()
running_loss = 0.0
log_steps = 0


while global_step < MAX_TRAIN_STEPS:

    # ========================================================
    # Learning rate for this step
    # ========================================================

    lr = get_lr(global_step)

    for group in optimizer.param_groups:
        group["lr"] = lr

    # ========================================================
    # Gradient accumulation
    # ========================================================

    optimizer.zero_grad(set_to_none=True)

    step_loss = 0.0

    for micro_step in range(GRAD_ACCUM_STEPS):

        xb, yb = next(train_iterator)

        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = train_model(xb, yb) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        step_loss += loss.item()

    # ========================================================
    # Clip gradients + optimizer step
    # ========================================================

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()

    global_step += 1
    running_loss += step_loss
    log_steps += 1

    # ========================================================
    # Logging
    # ========================================================

    if global_step % LOG_INTERVAL == 0:

        elapsed = time.time() - log_start
        tokens_per_sec = log_steps * TOKENS_PER_STEP / elapsed
        hours_left = (MAX_TRAIN_STEPS - global_step) * (elapsed / log_steps) / 3600

        print(
            f"Step {global_step:5d}/{MAX_TRAIN_STEPS} | "
            f"loss={running_loss / log_steps:.4f} | "
            f"lr={lr:.2e} | "
            f"grad_norm={grad_norm.item():.2f} | "
            f"{tokens_per_sec:,.0f} tok/s | "
            f"ETA {hours_left:.1f}h"
        )

        running_loss = 0.0
        log_steps = 0
        log_start = time.time()

    # ========================================================
    # Validation + checkpoints
    # ========================================================

    if global_step % EVAL_INTERVAL == 0 or global_step == MAX_TRAIN_STEPS:

        val_loss = evaluate_model(model, val_loader, MAX_VAL_BATCHES)

        print(
            f"\n[Validation] step={global_step} | "
            f"val_loss={val_loss:.4f} | "
            f"perplexity={math.exp(val_loss):.2f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CHECKPOINT, global_step, best_val_loss)
            print("⭐ New best checkpoint!")

        save_checkpoint(LAST_CHECKPOINT, global_step, best_val_loss)
        print()

    # ========================================================
    # Kaggle time limit
    # ========================================================

    if time.time() - session_start > TIME_LIMIT_HOURS * 3600:

        save_checkpoint(LAST_CHECKPOINT, global_step, best_val_loss)

        print(
            "\n⏰ Time limit reached. Attach this notebook's output, "
            "set RESUME_FROM to its last.pt and run again to continue."
        )
        break


# in case training stopped before the first validation
if not BEST_CHECKPOINT.exists():
    save_checkpoint(BEST_CHECKPOINT, global_step, best_val_loss)

print("\n✅ Training stopped at step", global_step)
print("Best validation loss:", best_val_loss)

TRAINING: step 0 -> 6000
Step    10/6000 | loss=7.8410 | lr=2.00e-05 | grad_norm=7.81 | 11,474 tok/s | ETA 9.5h
Step    20/6000 | loss=6.0701 | lr=4.00e-05 | grad_norm=4.09 | 10,996 tok/s | ETA 9.9h
Step    30/6000 | loss=5.5705 | lr=6.00e-05 | grad_norm=11.71 | 11,119 tok/s | ETA 9.8h
Step    40/6000 | loss=5.2783 | lr=8.00e-05 | grad_norm=1.89 | 11,144 tok/s | ETA 9.7h
Step    50/6000 | loss=4.7952 | lr=1.00e-04 | grad_norm=2.21 | 11,111 tok/s | ETA 9.7h
Step    60/6000 | loss=4.3787 | lr=1.20e-04 | grad_norm=1.18 | 11,160 tok/s | ETA 9.7h
Step    70/6000 | loss=4.0985 | lr=1.40e-04 | grad_norm=1.61 | 11,112 tok/s | ETA 9.7h
Step    80/6000 | loss=3.9112 | lr=1.60e-04 | grad_norm=1.39 | 11,113 tok/s | ETA 9.7h
Step    90/6000 | loss=3.7166 | lr=1.80e-04 | grad_norm=1.09 | 11,142 tok/s | ETA 9.7h
Step   100/6000 | loss=3.5683 | lr=2.00e-04 | grad_norm=1.78 | 11,143 tok/s | ETA 9.6h
Step   110/6000 | loss=3.4511 | lr=2.20e-04 | grad_norm=2.20 | 11,131 tok/s | ETA 9.6h
Step   120/6000 |

### CELL 17 · Load the best checkpoint

**In simple words:** Load the checkpoint with the lowest validation loss (1.0066, perplexity 2.74).

In [17]:
# ============================================================
# CELL 17 — LOAD BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best checkpoint")
print("Best step           :", checkpoint["step"])
print("Best validation loss:", checkpoint["best_val_loss"])

del checkpoint

Loaded best checkpoint
Best step           : 6000
Best validation loss: 1.0066178858280181


### CELL 18 · Generation

**In simple words:** Text generation: predict a token, append it, repeat. A repetition penalty and top-k sampling keep the text varied.

**How it works:** KV cache (earlier tokens are not recomputed); special tokens are never generated.

In [18]:
# ============================================================
# CELL 18 — GENERATION (with KV cache)
# ============================================================

# never generate these tokens
BANNED_IDS = [PAD_ID, UNK_ID, BOS_ID, EOS_ID]


@torch.no_grad()
def sample_next_token(logits, previous_ids, generator):

    logits = logits.float().clone()

    logits[BANNED_IDS] = -float("inf")

    # --------------------------------------------------------
    # Repetition penalty: make already-used tokens less likely
    # --------------------------------------------------------

    if GEN_REPETITION_PENALTY != 1.0:

        used = torch.tensor(sorted(set(previous_ids)), device=logits.device)
        used_logits = logits[used]

        logits[used] = torch.where(
            used_logits > 0,
            used_logits / GEN_REPETITION_PENALTY,
            used_logits * GEN_REPETITION_PENALTY,
        )

    logits = logits / GEN_TEMPERATURE

    # --------------------------------------------------------
    # Top-k: keep only the k most likely tokens
    # --------------------------------------------------------

    kth_value = torch.topk(logits, GEN_TOP_K).values[-1]
    logits[logits < kth_value] = -float("inf")

    # --------------------------------------------------------
    # Top-p: keep the smallest set with probability >= p
    # --------------------------------------------------------

    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)

    remove = sorted_probs.cumsum(dim=-1) - sorted_probs > GEN_TOP_P
    sorted_logits[remove] = -float("inf")

    logits = torch.full_like(logits, -float("inf"))
    logits[sorted_indices] = sorted_logits

    probs = F.softmax(logits, dim=-1)

    return torch.multinomial(probs, num_samples=1, generator=generator).item()


@torch.no_grad()
def generate_continuation(prompt, seed, target_words=GEN_TARGET_WORDS,
                          max_new_tokens=GEN_MAX_NEW_TOKENS):

    model.eval()

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    # --------------------------------------------------------
    # Encode prompt (keep the end if it is too long)
    # --------------------------------------------------------

    prompt_ids = tokenizer.encode(prompt)

    if len(prompt_ids) == 0:
        prompt_ids = [EOS_ID]          # empty prompt = start of a new document

    prompt_ids = prompt_ids[-(BLOCK_SIZE - max_new_tokens):]

    # one empty cache per layer
    kv_caches = [{"k": None, "v": None} for _ in range(MODEL_CONFIG["n_layers"])]

    input_ids = torch.tensor([prompt_ids], device=DEVICE)
    position = 0
    generated = []

    for _ in range(max_new_tokens):

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(input_ids, kv_caches=kv_caches, start_pos=position)

        position += input_ids.shape[1]

        next_token = sample_next_token(logits[0], prompt_ids + generated, generator)
        generated.append(next_token)

        # stop once word number (target_words + 1) has started,
        # so the first target_words words are complete
        words = tokenizer.decode(generated).split()

        if len(words) > target_words:
            return " ".join(words[:target_words])

        # next step: feed only the new token (older ones are in the cache)
        input_ids = torch.tensor([[next_token]], device=DEVICE)

    words = tokenizer.decode(generated).split()

    return " ".join(words[:target_words])

### CELL 19 · Manual generation test

**In simple words:** Try one prompt by hand.

**How it works:** Output removed before publishing.

In [19]:
# ============================================================
# CELL 19 — MANUAL GENERATION TEST
# ============================================================

TEST_PROMPT = (
    "The issue involved in these petitions is as to whether "
    "the court has jurisdiction to consider the matter under "
    "the applicable statutory provisions"
)

start = time.time()

generated_text = generate_continuation(TEST_PROMPT, seed=SEED)

print("PROMPT:\n", TEST_PROMPT)
print("\nGENERATED CONTINUATION:\n", generated_text)
print(f"\nWord count: {len(generated_text.split())} | time: {time.time() - start:.1f}s")

[output removed before publishing: this cell printed raw dataset / test text]


### CELL 20 · Download the test prompts

**In simple words:** Download the 1,000 Phase-1 test prompts.

In [20]:
# ============================================================
# CELL 20 — DOWNLOAD PHASE-1 TEST DATA
# (needs the Kaggle secret "HF" attached: Add-ons -> Secrets)
# ============================================================

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")

test_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test prompts:", len(test_data))

first_key = next(iter(test_data))
print("\nFirst test ID:", first_key)
print("First prompt :", test_data[first_key]["text"][:500])

Number of test prompts: 1000

First test ID: 1


### CELL 21 · Generate the test predictions

**In simple words:** Continue every test prompt (exactly 128 words each).

In [21]:
# ============================================================
# CELL 21 — GENERATE TEST PREDICTIONS
# ============================================================

test_items = list(test_data.items())

if DEBUG_MODE:
    test_items = test_items[:DEBUG_TEST_EXAMPLES]

submission = {}

start = time.time()

for i, (key, item) in enumerate(test_items, start=1):

    continuation = generate_continuation(item["text"], seed=SEED + i)

    # very rare: not enough words -> try again with more tokens
    if len(continuation.split()) < GEN_TARGET_WORDS:
        continuation = generate_continuation(
            item["text"], seed=SEED + i + 100_000,
            max_new_tokens=BLOCK_SIZE // 2,
        )

    submission[str(key)] = {"text": continuation}

    if i % 25 == 0 or i == len(test_items):
        minutes = (time.time() - start) / 60
        print(f"[{i}/{len(test_items)}] {minutes:.1f} min")

print("\n✅ Generation finished.")

[25/1000] 2.1 min
[50/1000] 4.1 min
[75/1000] 6.2 min
[100/1000] 8.3 min
[125/1000] 10.4 min
[150/1000] 12.5 min
[175/1000] 14.5 min
[200/1000] 16.9 min
[225/1000] 19.0 min
[250/1000] 21.1 min
[275/1000] 23.2 min
[300/1000] 25.2 min
[325/1000] 27.3 min
[350/1000] 29.3 min
[375/1000] 31.4 min
[400/1000] 33.5 min
[425/1000] 35.8 min
[450/1000] 37.8 min
[475/1000] 39.9 min
[500/1000] 42.0 min
[525/1000] 44.1 min
[550/1000] 46.1 min
[575/1000] 48.2 min
[600/1000] 50.3 min
[625/1000] 52.4 min
[650/1000] 54.4 min
[675/1000] 56.5 min
[700/1000] 58.8 min
[725/1000] 60.8 min
[750/1000] 62.9 min
[775/1000] 65.0 min
[800/1000] 67.3 min
[825/1000] 69.4 min
[850/1000] 71.4 min
[875/1000] 73.5 min
[900/1000] 75.8 min
[925/1000] 77.8 min
[950/1000] 79.9 min
[975/1000] 82.0 min
[1000/1000] 84.1 min

✅ Generation finished.


### CELL 22 · Validate + write the submission

**In simple words:** Check every prompt has an answer of 128 words, then write `prediction.json` + `submission.zip`.

In [22]:
# ============================================================
# CELL 22 — VALIDATE + SAVE prediction.json + SUBMISSION ZIP
# ============================================================

expected_keys = {str(key) for key, _ in test_items}

assert set(submission.keys()) == expected_keys, "Prediction IDs do not match test IDs"

for key, item in submission.items():
    word_count = len(item["text"].split())
    assert word_count == GEN_TARGET_WORDS, f"ID {key}: {word_count} words instead of {GEN_TARGET_WORDS}"

print(f"✅ All {len(submission)} predictions have exactly {GEN_TARGET_WORDS} words.")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)


ZIP_FILE = OUTPUT_DIR / "submission.zip"

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")


print("Prediction file:", PREDICTION_FILE)
print("Submission zip :", ZIP_FILE)

first_key = next(iter(submission))
print("\nExample:", json.dumps({first_key: submission[first_key]}, ensure_ascii=False, indent=2))

✅ All 1000 predictions have exactly 128 words.
Prediction file: /kaggle/working/legal_lm/prediction.json
Submission zip : /kaggle/working/legal_lm/submission.zip

Example: {
  "1": {
  }
}
